# Crawl Data Twitter > 2000 Tweets
The crawling process was done using Tweet-Harvest. Written by Helmi Satria on  March 30th 2024.



In [2]:
twitter_auth_token = '6c240dc7aee76e0d79ab72a3e29a19b39fa7daf6' # Token Anda

# Import library
import time
import pandas as pd
import os

# Install tweet-harvest (Cukup dijalankan sekali saja sebenarnya)
# Gunakan try-except agar tidak error jika sudah terinstall
#try:
#    !npm install -g tweet-harvest
#except:
#    print("Tweet harvest mungkin sudah terinstall atau npm tidak ditemukan.")

## Crawl Data - Kelas Normal

In [1]:
# Folder ini tempat hasil crawling akan disimpan
DATA_FOLDER = r"C:/Users/mjova/OneDrive/Documents/kuliah/Skripsi/deteksi_depresi/Data/Raw"

if not os.path.exists(DATA_FOLDER):
    os.makedirs(DATA_FOLDER)

crawl_months = [
    ("2025-07-01", "2025-07-31", "July"),
    ("2025-08-01", "2025-08-31", "August"),
    ("2025-09-01", "2025-09-30", "September"),
    ("2025-10-01", "2025-10-31", "October"),
    ("2025-11-01", "2025-11-30", "November")
]

# Contoh untuk Kelas Normal (Lakukan hal sama untuk Moderate & Severe)
for start_date, end_date, month_name in crawl_months:
    # Simpan langsung ke folder Raw lokal Anda
    filename = f"{DATA_FOLDER}/Normal_{month_name}.csv"
    query_file = f"query_normal_{month_name}.txt"

    # Query Normal (Menggunakan query expanded yang kita bahas sebelumnya)
    query_positive = '(senang OR bahagia OR semangat OR enjoy OR fun OR seru OR nyaman OR lega OR asyik OR puas OR sukses)'
    query_situational = '((gym OR lari OR futsal OR "naik gunung" OR olahraga OR drakor OR film OR series OR konser) AND (capek OR lelah OR pegal OR nangis OR sedih OR terharu))'
    query_content = f'(aku OR saya OR gue OR gw OR "rasanya aku") AND ({query_positive} OR {query_situational}) since:{start_date} until:{end_date} lang:id -filter:links -is:retweet'

    with open(query_file, 'w', encoding='utf-8') as f:
        f.write(query_content)

    print(f"PROGRESS: Normal - {month_name}")
    
    # PERHATIKAN PERUBAHAN DI BAWAH INI:
    # 1. {filename} menggantikan "$filename"
    # 2. {query_file} menggantikan {query_file} (tetap sama)
    # 3. {twitter_auth_token} menggantikan {twitter_auth_token} (tetap sama)
    !npx -y tweet-harvest@2.6.1 -o "{filename}" -s "$(cat {query_file})" --tab "LATEST" -l 500 --token {twitter_auth_token}
    
    print(f"WAIT: Delay 10 menit...")
    time.sleep(600)

print("SUCCESS: Crawling Normal Selesai")

NameError: name 'os' is not defined

## Crawl Data - Kelas Moderate

In [ ]:
for start_date, end_date, month_name in crawl_months:
    filename = f"Moderate_{month_name}.csv"
    query_file = f"query_moderate_{month_name}.txt"

    query_content = f'(aku OR saya OR gue OR gw OR rasanya) AND (sedih OR capek OR lelah OR stres OR bosan OR "ga mood" OR "ga semangat" OR sendirian OR bingung OR jenuh OR kosong) since:{start_date} until:{end_date} lang:id -filter:links -is:retweet'

    with open(query_file, 'w', encoding='utf-8') as f:
        f.write(query_content)

    print(f"PROGRESS: Moderate - {month_name}")
    !npx -y tweet-harvest@2.6.1 -o "{filename}" -s "$(cat {query_file})" --tab "LATEST" -l 500 --token {twitter_auth_token}
    print(f"WAIT: Delay 10 menit...")
    time.sleep(600)

print("SUCCESS: Moderate selesai")

## Crawl Data - Kelas Severe


In [ ]:
for start_date, end_date, month_name in crawl_months:
    filename = f"Severe_{month_name}.csv"
    query_file = f"query_severe_{month_name}.txt"

    query_content = f'(aku OR saya OR gue OR gw) AND (pengen OR mau OR ingin OR "ga kuat" OR cape OR lelah) AND (mati OR "bunuh diri" OR "luka diri" OR hilang OR "berhenti hidup") since:{start_date} until:{end_date} lang:id -filter:links -is:retweet'

    with open(query_file, 'w', encoding='utf-8') as f:
        f.write(query_content)

    print(f"PROGRESS: Severe - {month_name}")
    !npx -y tweet-harvest@2.6.1 -o "{filename}" -s "$(cat {query_file})" --tab "LATEST" -l 300 --token {twitter_auth_token}
    print(f"WAIT: Delay 10 menit...")
    time.sleep(600)

print("SUCCESS: Severe selesai")

## Tahap Pembersihan & Filtering Data (Final Pipeline)

In [ ]:
# Setup dan Konfigurasi
import pandas as pd
import glob
import re
from collections import Counter
import os
import time

# Konfigurasi folder - GANTI PATH SESUAI LOKASI Anda
possible_folders = [
    "deteksi_depresi/Data/Raw",       # Jika dijalankan dari luar folder proyek
    "Data/Raw",                       # Jika dijalankan dari dalam folder deteksi_depresi
    "../Data/Raw",                        # Jika dijalankan dari dalam folder scripts
]

CRAWL_FOLDER = None
print("🔍 Mencari folder data raw...")
for folder in possible_folders:
    if glob.glob(f"{folder}/*.csv"):
        CRAWL_FOLDER = folder
        print(f"✅ Folder DITEMUKAN: {CRAWL_FOLDER}")
        break

if CRAWL_FOLDER is None:
    print("❌ ERROR: Folder data tidak ditemukan! Pastikan file CSV hasil crawling ada.")
    CRAWL_FOLDER = "." 

OUTPUT_FOLDER = "filtered-data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# --- 2. KONFIGURASI FILTER ---
MAX_TWEETS_PER_USER = 3  # Batas maksimal tweet per user agar data variatif
METADATA_COLUMNS = ['created_at', 'id_str', 'full_text', 'username', 'favorite_count', 'retweet_count', 'tweet_url']

print("Konfigurasi selesai.")

🔍 Mencari folder data raw...
✅ Folder DITEMUKAN: ../Data/Raw
Konfigurasi selesai.


In [3]:
# --- DEFINISI REGEX DIPERKAYA (ENHANCED) ---

POSITIVE_KEYWORDS = [
    r'senang', r'bahagia', r'semangat', r'enjoy', r'fun', r'seru', r'nyaman', 
    r'lega', r'asyik', r'puas', r'sukses', r'happy', r'seneng', r'bersyukur', 
    r'alhamdulillah', r'mantap', r'keren', r'lucu', r'ngakak', r'excited', 
    r'love', r'suka', r'terhibur', r'segar', r'sehat', r'produktif', r'aman',
    r'bangga', r'hebat', r'keren'
]
ROUTINE_CONTEXT = [
    r'kerja', r'kantor', r'lembur', r'meeting', r'rapat', r'job', r'gaji',
    r'kuliah', r'skripsi', r'tugas', r'ujian', r'sekolah', r'guru', r'dosen', r'kelas',
    r'macet', r'panas', r'hujan', r'antri', r'jalan', r'motor', r'mobil',
    r'drakor', r'film', r'nonton', r'game', r'mabar', r'konser', r'lagu',
    r'olahraga', r'gym', r'lari', 'futsal', 'basket', 'badminton',
    r'lapar', r'ngantuk', r'pegal', r'capek banget', r'pusing banget' 
]

# Keywords Moderate/Severe
NEGATIVE_STATE = [
    r'sedih', r'hampa', r'kosong', r'kesepian', r'sendirian', r'bingung', 
    r'jenuh', r'nangis', r'kecewa', r'sakit hati', r'putus asa', r'tidak berharga',
    r'beban', r'gagal', r'benci diri'
]
SEVERE_KEYWORDS = [
    r'mati', r'bunuh diri', r'luka diri', r'hilang', r'berhenti hidup', 
    r'akhiri hidup', r'tidur selamanya', r'ingin mati', r'pengen mati', 
    r'ga kuat hidup', r'mau mati'
]

# Blacklist Normal (Agar curhat mental tidak masuk Normal)
MENTAL_BLACKLIST = [
    r'mental', r'batin', r'jiwa', r'hati', r'perasaan', 
    r'nangis kejer', r'sesak', r'hancur', r'rapuh', 
    r'kena mental', r'sakit hati', r'muak', r'capek hati',
    r'lelah hati', r'lelah mental', r'capek mental'
]

# Regex Compiled
POSITIVE_REGEX = r'\b(' + '|'.join(POSITIVE_KEYWORDS) + r')\b'
ROUTINE_REGEX = r'\b(' + '|'.join(ROUTINE_CONTEXT) + r')\b'
NEGATIVE_REGEX = r'\b(' + '|'.join(NEGATIVE_STATE) + r')\b'
SEVERE_REGEX = r'\b(' + '|'.join(SEVERE_KEYWORDS) + r')\b'
MENTAL_BLACKLIST_REGEX = r'\b(' + '|'.join(MENTAL_BLACKLIST) + r')\b'
PRONOUN_REGEX = r'\b(aku|saya|gue|gw|w|ku|kita|q|ak|sy)\b'

# --- 4. FUNGSI VALIDASI ---
def validate_tweet(text, class_label):
    if not isinstance(text, str): return "INVALID"
    text_clean = text.lower().replace('\n', ' ') 
    
    has_pronoun = bool(re.search(PRONOUN_REGEX, text_clean))
    
    # Normal (0)
    if class_label == 0:
        if re.search(MENTAL_BLACKLIST_REGEX, text_clean): return "INVALID_MENTAL"
        has_positive = bool(re.search(POSITIVE_REGEX, text_clean))
        has_routine = bool(re.search(ROUTINE_REGEX, text_clean))
        if has_pronoun and (has_positive or has_routine): return "VALID"
        return "INVALID"

    # Moderate (1)
    if class_label == 1:
        has_negative = bool(re.search(NEGATIVE_REGEX, text_clean))
        has_severe = bool(re.search(SEVERE_REGEX, text_clean))
        if has_pronoun and has_negative and not has_severe: return "VALID"
        return "INVALID"

    # Severe (2)
    if class_label == 2:
        has_severe = bool(re.search(SEVERE_REGEX, text_clean))
        if has_severe: return "VALID"
        return "INVALID"
    
    return "INVALID"
print("Regex & Validasi Diperbarui (Enhanced Version).")

Regex & Validasi Diperbarui (Enhanced Version).


In [4]:
print("\n" + "="*60)
print("MEMULAI FILTERING (DENGAN METADATA LENGKAP)")
print("="*60)

all_valid_data = []
class_distribution = Counter()

for class_name, label in [("Normal", 0), ("Moderate", 1), ("Severe", 2)]:
    print(f"\n📂 Memproses kelas: {class_name}")
    
    pattern = f"{CRAWL_FOLDER}/*{class_name}*.csv"
    class_files = glob.glob(pattern)
    print(f"   Ditemukan {len(class_files)} file.")
    
    count_valid = 0
    
    for file in class_files:
        try:
            df = pd.read_csv(file, on_bad_lines='skip')
            
            # Cek kolom teks utama
            col_text = None
            for c in ['full_text', 'tweet', 'text']:
                if c in df.columns:
                    col_text = c
                    break
            
            if not col_text: continue
                
            for idx, row in df.iterrows():
                tweet = str(row.get(col_text, ''))
                status = validate_tweet(tweet, label)
                
                if status == "VALID":
                    # [PENTING] Ambil Metadata Lengkap
                    data_row = {
                        'label': label,
                        'class_name': class_name,
                        'source_file': os.path.basename(file),
                        'tweet': tweet  # Text utama untuk training
                    }
                    
                    # Masukkan kolom metadata lain jika ada di raw data
                    for meta_col in METADATA_COLUMNS:
                        if meta_col in df.columns and meta_col != col_text:
                            data_row[meta_col] = row.get(meta_col, '')
                    
                    # Fallback untuk id jika tidak ada
                    if 'id_str' not in data_row:
                        data_row['id_str'] = f"{label}_{idx}"

                    all_valid_data.append(data_row)
                    count_valid += 1

        except Exception as e:
            print(f"   ❌ Error: {e}")

    print(f"   ✅ Total Valid: {count_valid}")
    class_distribution[label] = count_valid

# --- 6. SIMPAN HASIL ---
if all_valid_data:
    df_valid = pd.DataFrame(all_valid_data)
    
    # Hapus duplikat berdasarkan ID atau Text
    if 'id_str' in df_valid.columns:
        df_valid.drop_duplicates(subset=['id_str'], inplace=True)
    else:
        df_valid.drop_duplicates(subset=['tweet'], inplace=True)
    
    # 1. Simpan ARSIP LENGKAP (Dengan Metadata)
    full_path = f"{OUTPUT_FOLDER}/Dataset_Full_Metadata.csv"
    df_valid.to_csv(full_path, index=False)
    print(f"\n✅ [ARSIP] Tersimpan: {full_path}")
    print(f"   Jumlah: {len(df_valid)} baris")
    print(f"   Kolom: {list(df_valid.columns)}")
    
    # 2. Simpan DATASET TRAINING (Hanya tweet & label)
    clean_path = f"{OUTPUT_FOLDER}/Dataset_Valid_Cleaned.csv"
    df_training = df_valid[['tweet', 'label']].copy()
    df_training.to_csv(clean_path, index=False)
    print(f"✅ [TRAINING] Tersimpan: {clean_path}")
    print(f"   Siap untuk train_model.py")
    
    print(f"\n📊 Distribusi Final: {dict(class_distribution)}")
else:
    print("\n⚠️ Data Valid Kosong.")


MEMULAI FILTERING (DENGAN METADATA LENGKAP)

📂 Memproses kelas: Normal
   Ditemukan 5 file.
   ✅ Total Valid: 3175

📂 Memproses kelas: Moderate
   Ditemukan 5 file.
   ✅ Total Valid: 2487

📂 Memproses kelas: Severe
   Ditemukan 5 file.
   ✅ Total Valid: 2555

✅ [ARSIP] Tersimpan: filtered-data/Dataset_Full_Metadata.csv
   Jumlah: 8140 baris
   Kolom: ['label', 'class_name', 'source_file', 'tweet', 'created_at', 'id_str', 'username', 'favorite_count', 'retweet_count', 'tweet_url']
✅ [TRAINING] Tersimpan: filtered-data/Dataset_Valid_Cleaned.csv
   Siap untuk train_model.py

📊 Distribusi Final: {0: 3175, 1: 2487, 2: 2555}


In [5]:
# --- GENERATE SAMPLE & REPORT ---

# 1. Buat Sample Review (100 data per kelas untuk dicek manual)
def generate_sample(df, label, sample_size=100):
    df_class = df[df['label'] == label]
    if len(df_class) < sample_size:
        return df_class
    return df_class.sample(n=sample_size, random_state=42)

if all_valid_data:
    print("\nMembuat Sample Review...")
    for label, class_name in [(0, "Normal"), (1, "Moderate"), (2, "Severe")]:
        sample_df = generate_sample(df_valid, label, 100)
        sample_df.to_csv(f"{OUTPUT_FOLDER}/Sample_Review_{class_name}.csv", index=False)
        print(f" -> {class_name}: {len(sample_df)} sample tersimpan.")

    # 2. Buat Laporan Teks
    report_path = f"{OUTPUT_FOLDER}/cleaning_report.txt"
    with open(report_path, "w") as f:
        f.write("CLEANING REPORT\n")
        f.write("="*30 + "\n")
        f.write(f"Total Data: {len(df_valid)}\n")
        f.write(f"Distribusi: {dict(class_distribution)}\n")
    print(f"\nLaporan tersimpan di: {report_path}")


Membuat Sample Review...
 -> Normal: 100 sample tersimpan.
 -> Moderate: 100 sample tersimpan.
 -> Severe: 100 sample tersimpan.

Laporan tersimpan di: filtered-data/cleaning_report.txt


In [6]:
#Generate Sample Review
print("\n" + "="*60)
print("GENERATING SAMPLE REVIEW")
print("="*60)

def generate_sample(df, label, sample_size=100):
    df_class = df[df['label'] == label]

    if len(df_class) < sample_size:
        print(f"  WARNING: Kelas {label} hanya punya {len(df_class)} tweet. Sample = semua data.")
        return df_class

    sample = []
    files = df_class['source_file'].unique()
    per_file = sample_size // len(files)

    for f in files:
        df_file = df_class[df_class['source_file'] == f]
        sample_file = df_file.sample(n=min(per_file, len(df_file)), random_state=42)
        sample.append(sample_file)

    sample_df = pd.concat(sample)
    if len(sample_df) < sample_size:
        remaining = df_class[~df_class.index.isin(sample_df.index)]
        additional = remaining.sample(n=sample_size-len(sample_df), random_state=42)
        sample_df = pd.concat([sample_df, additional])

    return sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

for label, class_name in [(0, "Normal"), (1, "Moderate"), (2, "Severe")]:
    sample_df = generate_sample(df_valid, label, 100)
    sample_file = f"{OUTPUT_FOLDER}/Sample_Review_{class_name}.csv"
    sample_df.to_csv(sample_file, index=False)
    print(f"  Sample_Review_{class_name}.csv: {len(sample_df)} tweet")

print("\nSample review siap untuk dilihat di folder:", OUTPUT_FOLDER)


GENERATING SAMPLE REVIEW
  Sample_Review_Normal.csv: 100 tweet
  Sample_Review_Moderate.csv: 100 tweet
  Sample_Review_Severe.csv: 100 tweet

Sample review siap untuk dilihat di folder: filtered-data


In [7]:
#Lihat Sample
sample_normal = pd.read_csv(f"{OUTPUT_FOLDER}/Sample_Review_Normal.csv")
print("=== SAMPLE NORMAL (10 pertama) ===")
for i, row in sample_normal.head(10).iterrows():
    print(f"{i+1}. {row['tweet'][:80]}...")

sample_moderate = pd.read_csv(f"{OUTPUT_FOLDER}/Sample_Review_Moderate.csv")
print("\n=== SAMPLE MODERATE (10 pertama) ===")
for i, row in sample_moderate.head(10).iterrows():
    print(f"{i+1}. {row['tweet'][:80]}...")

sample_severe = pd.read_csv(f"{OUTPUT_FOLDER}/Sample_Review_Severe.csv")
print("\n=== SAMPLE SEVERE (10 pertama) ===")
for i, row in sample_severe.head(10).iterrows():
    print(f"{i+1}. {row['tweet'][:80]}...")

=== SAMPLE NORMAL (10 pertama) ===
1. cuma gue jawab jihad orang beda2 nte anakmu jihad pke puasa daud aku jihad dgn k...
2. Capek bgt akhir2 ini. Gila ya gw cuti sakit orang2 dikantor minim kerja. Giliran...
3. capek kerja gaji buat kebutuhan sma byr pinjol. Yallah kapan part aku bisa menik...
4. @poconxg Capek banget pengen kerja pengen tinggal jauh dari keluarga lagi. Rasan...
5. gue sedih trs milih jogging tp cmn berapa puteran (at least ga kepikiran yg biki...
6.  Hey aku lg di sby nih Mas aku di sby mau cari sesuatu temenin ya Gak gak aku ke...
7. nggak ada kerjaan tidak menghentikqn aku buat nangis di toilet kantor (ritual)...
8. capek ngobrol sama orang gila lama² ikutan gila kok bisaaa gitu ada energinya bu...
9. Gue mau nangis aja bru dpt kabar ziel udh balik lg kerja di LN ga ngabarin gwej....
10. Capek batt ga berenti nangis dari tadi tiap videonya lewatini harusnya kantor gu...

=== SAMPLE MODERATE (10 pertama) ===
1. @tanyarlfes hah kak aku malah gabisa ngerasain patah ha

## Laporan Akhir

In [18]:
# --- LAPORAN AKHIR (OTOMATIS DARI FILE RAW) ---
print("\n" + "="*60)
print("GENERATE CLEANING REPORT")
print("="*60)

report = [
    "CLEANING REPORT - DETEKSI DEPRESI",
    "="*50,
    f"Total tweet valid (Final): {len(df_valid)}",
    f"Distribusi kelas final: {dict(class_distribution)}",
    "-"*50
]

for class_name, label in [("Normal", 0), ("Moderate", 1), ("Severe", 2)]:
    # 1. Hitung Data Final (Setelah Filter)
    final_count = len(df_valid[df_valid['label'] == label])

    # 2. Hitung Data Mentah (Raw) Langsung dari File
    # Cari semua file raw untuk kelas ini
    raw_files = glob.glob(f"{CRAWL_FOLDER}/{class_name}_*.csv")
    total_raw = 0

    for f in raw_files:
        try:
            # Baca file hanya untuk hitung panjangnya
            # Gunakan engine='python' dan sep=None agar otomatis deteksi koma/titik koma
            df_temp = pd.read_csv(f, on_bad_lines='skip', sep=None, engine='python')
            total_raw += len(df_temp)
        except:
            continue # Abaikan jika file rusak/kosong

    # 3. Hitung Persentase Pengurangan
    # Hindari pembagian dengan nol
    if total_raw > 0:
        reduction_percentage = ((total_raw - final_count) / total_raw) * 100
    else:
        reduction_percentage = 0

    report.append(f"KELAS: {class_name}")
    report.append(f"  - Total Raw Data (Crawling): {total_raw}")
    report.append(f"  - Total Valid Data (Clean):  {final_count}")
    report.append(f"  - Data Dibuang (Filter):     {reduction_percentage:.1f}%")
    report.append("")

report.append("Catatan:")
report.append("- Total Raw Data dihitung real-time dari jumlah baris file CSV di folder Raw.")
report.append("- Data Dibuang meliputi: Duplikat ID, Spam User, dan Teks Tidak Relevan (Regex).")

# Simpan Laporan
with open(f"{OUTPUT_FOLDER}/cleaning_report.txt", "w") as f:
    f.write("\n".join(report))

print("\n".join(report))
print(f"\nLaporan tersimpan di: {OUTPUT_FOLDER}/cleaning_report.txt")


GENERATE CLEANING REPORT
CLEANING REPORT - DETEKSI DEPRESI
Total tweet valid (Final): 7568
Distribusi kelas final: {0: 2644, 1: 2420, 2: 2549}
--------------------------------------------------
KELAS: Normal
  - Total Raw Data (Crawling): 2784
  - Total Valid Data (Clean):  2622
  - Data Dibuang (Filter):     5.8%

KELAS: Moderate
  - Total Raw Data (Crawling): 2526
  - Total Valid Data (Clean):  2400
  - Data Dibuang (Filter):     5.0%

KELAS: Severe
  - Total Raw Data (Crawling): 2554
  - Total Valid Data (Clean):  2546
  - Data Dibuang (Filter):     0.3%

Catatan:
- Total Raw Data dihitung real-time dari jumlah baris file CSV di folder Raw.
- Data Dibuang meliputi: Duplikat ID, Spam User, dan Teks Tidak Relevan (Regex).

Laporan tersimpan di: filtered-data/cleaning_report.txt
